# Step 2 · Preprocessing — QA report

Step 2 is run by the two scripts (in this order):

```bash
python -m src.step2_preprocess       # step1_corpus.csv        -> step2_chunks_clean.csv
python -m src.step2_keyword_filter   # step2_chunks_clean.csv  -> step2_filtered_corpus.csv
```

The logic lives in `src/step2_preprocess.py` (clean + chunk) and `src/step2_keyword_filter.py`
(keep only chunks matching the reputation-risk dictionary). This notebook loads their outputs
and checks the results, so it can be re-run any time without side effects.

In [1]:
# Auto-reload edited src/ modules without restarting the kernel
%load_ext autoreload
%autoreload 2

import ast
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src import step2_preprocess as preprocess
from src import step2_keyword_filter as keyword_filter

for p in (preprocess.OUTPUT_CSV, keyword_filter.OUTPUT_PATH):
    assert p.exists(), f'{p.name} missing - run the step 2 scripts first (see cell above).'

chunks   = pd.read_csv(preprocess.OUTPUT_CSV)
filtered = pd.read_csv(keyword_filter.OUTPUT_PATH)
filtered['matched_categories'] = filtered['matched_categories'].apply(ast.literal_eval)
print(f'Loaded {len(chunks):,} chunks and {len(filtered):,} filtered chunks')

Loaded 76,581 chunks and 14,939 filtered chunks


## 1 · Overview of the chunks

In [2]:
n_filings = chunks.groupby(['cik', 'year']).ngroups
print(f'Chunks          : {len(chunks):,}')
print(f'Filings         : {n_filings:,}')
print(f'Mean chunks/doc : {len(chunks)/n_filings:.1f}')
print(f'Companies       : {chunks["cik"].nunique()}')
print(f'Sectors         : {chunks["sector"].nunique()}')
print('\nChunk word-count distribution:')
print(chunks['n_words'].describe().round(1).to_string())

Chunks          : 76,581
Filings         : 1,661
Mean chunks/doc : 46.1
Companies       : 109
Sectors         : 11

Chunk word-count distribution:
count    76581.0
mean       192.0
std        144.7
min         20.0
25%         56.0
50%        140.0
75%        370.0
max        858.0


## 2 · Quality assurance (chunking)

In [3]:
problems = {}
problems['empty text']            = int((chunks['text'].str.strip() == '').sum())
problems['below MIN_CHUNK_WORDS'] = int((chunks['n_words'] < preprocess.MIN_CHUNK_WORDS).sum())
problems['above MAX_CHUNK_WORDS'] = int((chunks['n_words'] > preprocess.MAX_CHUNK_WORDS).sum())
problems['missing sector']        = int(chunks['sector'].isna().sum())
problems['starts lowercase/punct']= int((~chunks['text'].str.match(r'^[A-Z0-9"]')).sum())

for k, v in problems.items():
    flag = 'OK' if v == 0 else 'CHECK'
    print(f'  [{flag}] {k}: {v}')

  [OK] empty text: 0
  [OK] below MIN_CHUNK_WORDS: 0
  [CHECK] above MAX_CHUNK_WORDS: 140
  [OK] missing sector: 0
  [CHECK] starts lowercase/punct: 3910


## 3 · Chunks per year and per sector

In [4]:
print('Chunks per year:')
print(chunks.groupby('year').size().to_string())
print('\nChunks per sector:')
print(chunks.groupby('sector').size().sort_values(ascending=False).to_string())

Chunks per year:
year
2010    8054
2011    7162
2012    7391
2013    5878
2014    4454
2015    4246
2016    3876
2017    4004
2018    3992
2019    3962
2020    3729
2021    3893
2022    3846
2023    3732
2024    4112
2025    4250

Chunks per sector:
sector
Real Estate               10007
Health Care                9257
Utilities                  8739
Financials                 7895
Information Technology     7617
Consumer Discretionary     7168
Communication Services     6900
Materials                  5854
Industrials                5004
Energy                     4606
Consumer Staples           3534


## 4 · Sample chunks

In [5]:
for _, r in chunks.sample(3, random_state=0).iterrows():
    print(f"\n=== {r['ticker']} {r['year']} · chunk {r['chunk_id']} · {r['sector']} ({r['n_words']}w) ===")
    print('TEXT:', r['text'][:300])


=== WMB 2010 · chunk 35 · Energy (129w) ===
TEXT: Our Gas Pipeline and Midstream businesses provide some services pursuant to long-term, fixed price contracts. It is possible that costs to perform services under such contracts will exceed the revenues we collect for our services. Although most of the services provided by our interstate gas pipeline

=== PNW 2022 · chunk 9 · Utilities (393w) ===
TEXT: This is in large part due to a 2004 Arizona Court of Appeals decision that found critical components of the ACC’s rules to be violative of the Arizona Constitution. The ruling also voided the operating authority of all the competitive providers previously authorized by the ACC. On May 9, 2013, the A

=== COP 2022 · chunk 94 · Energy (36w) ===
TEXT: In addition, although we anticipate we will be able to repay our existing indebtedness when it matures or in accordance with our stated plans, there can be no assurance we will be able to do so.


## 5 · Keyword filter QA

How much of the corpus survives the reputation-risk filter, and which categories match.

In [6]:
print(f'Chunks in : {len(chunks):,}')
print(f'Chunks out: {len(filtered):,}  ({len(filtered)/len(chunks)*100:.1f}% retained)')
dups = filtered.duplicated(subset=['text', 'ticker', 'year']).sum()
print(f'Exact duplicates in filtered corpus: {dups}')

categories = keyword_filter.load_categories(keyword_filter.KEYWORDS_PATH)
print('\nMatches per category:')
counts = {cat: int(filtered['matched_categories'].apply(lambda c: cat in c).sum()) for cat in categories}
for cat, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    flag = '  <-- ZERO HITS' if n == 0 else ''
    print(f'  {n:6d}  {cat}{flag}')

Chunks in : 76,581
Chunks out: 14,939  (19.5% retained)
Exact duplicates in filtered corpus: 16

Matches per category:
    3681  Cybersecurity & Data Privacy
    2691  Governance Risk
    2665  Trademark/Brand Erosion
    2308  Credit, Liquidity & Market Risk
    1860  Financial Performance & Fraud
    1769  Legal & Litigation
    1536  Environmental Misconduct
    1090  Product/Service Quality
     659  Production Failure
     648  Health & Safety
     315  Labor/Human Rights
     253  Human Resources
      51  Supply Chain & Third-Party Risk
      18  Data/Privacy Breaches


## 6 · Sample filtered chunks

In [7]:
for _, r in filtered.sample(2, random_state=0).iterrows():
    print(f"\n=== {r['ticker']} {r['year']} · {r['matched_categories']} ===")
    print('TEXT:', r['text'][:300])


=== WELL 2025 · ['Environmental Misconduct'] ===
TEXT: As a result, if a liability were asserted against us based on ownership of those properties, we might have to pay substantial sums to settle or contest it, which could adversely affect our results of operations and cash flow. Unknown liabilities with respect to acquired properties might include, amo

=== LYV 2025 · ['Cybersecurity & Data Privacy'] ===
TEXT: We have expended significant capital and other resources to protect against and remedy such potential security breaches, incidents and their consequences, including the establishment of a dedicated cybersecurity organization within our larger technology environment, and will continue to do so in the


---
Next: `python src/step3_model.py` trains BERTopic on `step2_filtered_corpus.csv`.